<a href="https://colab.research.google.com/github/Reginajose/Observatorio-Clima-Grande-desafio/blob/main/Analise_dos_Dados_do_Atlas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise Exploratória e Diagnóstico de Qualidade (Sprint 1)
**Objetivo:** Este caderno (notebook) não altera nem corrige os dados. O objetivo aqui é apenas "tirar uma radiografia" da base de dados do Atlas Digital de Desastres.

Vamos carregar o arquivo original, aplicar o filtro do nosso escopo (Região Sul, 2021 a 2025, Desastres Hidrológicos) e gerar um relatório apontando o que está consistente e o que tem problemas (dados nulos, em branco ou mal formatados), para documentar no nosso Data Card.

In [ ]:
# Importamos a biblioteca 'pandas', que é a ferramenta padrão do Python para ler e analisar tabelas
import pandas as pd

# 1. Carregamento da base de dados
# Indicamos o nome exato do arquivo. O 'sep=";"' avisa ao computador que as colunas no Brasil são separadas por ponto e vírgula
caminho_arquivo = 'BD_Atlas_1991_2025_v1.1_2026.08.06_Consolidado.csv'

print("Lendo o arquivo original...")
df_bruto = pd.read_csv(caminho_arquivo, sep=';', low_memory=False)

print(f"Base carregada com sucesso! O arquivo inteiro tem {df_bruto.shape[0]} linhas e {df_bruto.shape[1]} colunas.")

Lendo o arquivo original...
Base carregada com sucesso! O arquivo inteiro tem 76190 linhas e 70 colunas.


## 1. Aplicando o Recorte do Projeto
A base contém o Brasil inteiro desde 1991. Para avaliar a qualidade do dado que *nós* vamos usar, primeiro precisamos isolar apenas o nosso alvo: estados do Sul (RS, SC, PR), anos de 2021 a 2025, e o grupo Hidrológico (que engloba Inundações, Enxurradas, Alagamentos, Chuvas Intensas e Movimento de Massa).

In [ ]:
# O Python precisa entender que a coluna "Data_Evento" é uma data real (dia/mês/ano), e não apenas um texto.
# O comando 'coerce' avisa: se achar uma data impossível (como 32/13/2021), transforme em um valor nulo (NaT) para acusarmos o erro depois.
df_bruto['Data_Evento_dt'] = pd.to_datetime(df_bruto['Data_Evento'], format='%d/%m/%Y', errors='coerce')

# Criamos as "regras" (filtros) que queremos aplicar na tabela
filtro_estado = df_bruto['Sigla_UF'].isin(['PR', 'RS', 'SC'])
filtro_ano = (df_bruto['Data_Evento_dt'].dt.year >= 2021) & (df_bruto['Data_Evento_dt'].dt.year <= 2025)
filtro_grupo = df_bruto['grupo_de_desastre'] == 'Hidrológico'

# Criamos uma nova tabela (df_recorte) aplicando todas as regras ao mesmo tempo
df_recorte = df_bruto[filtro_estado & filtro_ano & filtro_grupo].copy()

print(f"Pronto! Nosso recorte tem exatamente {len(df_recorte)} registros de desastres.")

Pronto! Nosso recorte tem exatamente 3756 registros de desastres.


## 2. Estado Atual da Base (Distribuição Geográfica e Tipologias)
Agora vamos pedir para o computador contar quantas linhas existem para cada estado, quais os tipos exatos de desastres que sobraram no recorte e qual o status de reconhecimento deles no sistema S2iD.

In [ ]:
# O comando 'value_counts()' agrupa os itens iguais e conta quantos existem de cada
print("--- DISTRIBUIÇÃO POR ESTADO ---")
print(df_recorte['Sigla_UF'].value_counts())

print("\n--- QUAIS TIPOS DE DESASTRES EXISTEM NESTE RECORTE? ---")
print(df_recorte['descricao_tipologia'].value_counts())

print("\n--- STATUS LEGAL DOS DESASTRES ---")
# Mostra se o desastre foi apenas registrado pela prefeitura ou se já foi reconhecido oficialmente pelo Governo
print(df_recorte['Status'].value_counts())

--- DISTRIBUIÇÃO POR ESTADO ---
Sigla_UF
SC    1989
RS    1517
PR     250
Name: count, dtype: int64

--- QUAIS TIPOS DE DESASTRES EXISTEM NESTE RECORTE? ---
descricao_tipologia
Chuvas Intensas       2630
Enxurradas             609
Alagamentos            220
Inundações             192
Movimento de Massa     105
Name: count, dtype: int64

--- STATUS LEGAL DOS DESASTRES ---
Status
Registro       1984
Reconhecido    1772
Name: count, dtype: int64


## 3. Diagnóstico de Qualidade: Identificando Valores Nulos (Vazios)
Um dos maiores problemas em análise de dados é a falta de preenchimento. Se colunas vitais como o Código IBGE, Danos Materiais ou População Afetada estiverem em branco, nosso cruzamento com o MapBiomas vai falhar. Aqui vamos apenas contar e expor quantos buracos existem na base.

In [ ]:
# Selecionamos as colunas que consideramos obrigatórias para o nosso projeto funcionar
colunas_vitais = [
    'Cod_IBGE_Mun',
    'Nome_Municipio',
    'Data_Evento',
    'DM_total_danos_materiais', # Obras destruídas
    'PEPL_total_publico',       # Serviços de emergência paralisados
    'DH_MORTOS',
    'DH_DESALOJADOS'
]

# O comando 'isnull().sum()' verifica linha por linha nessas colunas e soma quantas estão totalmente vazias
nulos_encontrados = df_recorte[colunas_vitais].isnull().sum()

print("--- RELATÓRIO DE BURACOS (VALORES NULOS) ---")
print(nulos_encontrados)

# Vamos checar especificamente se alguma data falhou na conversão inicial (datas inválidas)
datas_invalidas = df_recorte['Data_Evento_dt'].isnull().sum()
print(f"\nExistem {datas_invalidas} registros com datas impossíveis de ler pelo computador.")

--- RELATÓRIO DE BURACOS (VALORES NULOS) ---
Cod_IBGE_Mun                0
Nome_Municipio              0
Data_Evento                 0
DM_total_danos_materiais    0
PEPL_total_publico          0
DH_MORTOS                   0
DH_DESALOJADOS              0
dtype: int64

Existem 0 registros com datas impossíveis de ler pelo computador.


## 4. Diagnóstico de Qualidade: Identificando Dados Sujos (Inconsistências)
Às vezes o dado não está nulo, mas está digitado errado. Um erro clássico de prefeituras é colocar um espaço em branco no final do nome da cidade (ex: digitar `"Curitiba "` em vez de `"Curitiba"`). O olho humano não vê diferença, mas o computador considera que são duas cidades diferentes. Vamos rastrear isso.

In [ ]:
# O código abaixo cria uma versão invisível da coluna de municípios removendo espaços inúteis no início e no fim (.str.strip())
# Depois, ele compara a versão limpa com a versão original. Se forem diferentes, é porque o original estava sujo.
municipios_originais = df_recorte['Nome_Municipio'].astype(str)
municipios_sem_espaco = municipios_originais.str.strip()

# Contamos quantas vezes o original é diferente do limpo
quantidade_sujos = (municipios_originais != municipios_sem_espaco).sum()

print("--- RELATÓRIO DE DADOS SUJOS (ESPAÇOS INVISÍVEIS) ---")
print(f"Foram encontrados {quantidade_sujos} registros onde o nome do município tem espaços em branco extras sobrando.")

--- RELATÓRIO DE DADOS SUJOS (ESPAÇOS INVISÍVEIS) ---
Foram encontrados 0 registros onde o nome do município tem espaços em branco extras sobrando.


## 5. Resumo das Colunas Financeiras e de Impacto
Por fim, vamos olhar para o panorama dos valores declarados (em Reais e em Vidas). O comando `describe()` nos mostra qual foi a média de prejuízo por desastre, qual foi o valor máximo declarado, se existem valores negativos (o que seria um erro grave) e como os números se distribuem.

In [ ]:
# Separamos as colunas que representam dinheiro e impacto social
colunas_impacto = [
    'DM_total_danos_materiais',
    'PEPL_total_publico',
    'DH_MORTOS',
    'DH_DESALOJADOS',
    'DH_DESABRIGADOS'
]

# Forçamos o computador a ler essas colunas como números (caso alguma prefeitura tenha digitado letras por engano)
# O 'errors=coerce' transforma letras perdidas em nulos (NaN) para não quebrar a conta
df_impacto_numerico = df_recorte[colunas_impacto].apply(pd.to_numeric, errors='coerce')

# Tiramos a notação científica (aqueles números com "e+06") para formatar como moeda/número real com 2 casas decimais
pd.options.display.float_format = '{:,.2f}'.format

print("--- ESTATÍSTICAS DOS DANOS E PREJUÍZOS ---")
print(df_impacto_numerico.describe())

--- ESTATÍSTICAS DOS DANOS E PREJUÍZOS ---
       DM_total_danos_materiais  PEPL_total_publico  DH_MORTOS  \
count                  3,756.00            3,756.00   3,756.00   
mean               3,673,486.23          742,814.48       0.08   
std               81,223,851.76        4,115,223.16       0.78   
min                        0.00                0.00       0.00   
25%                        0.00                0.00       0.00   
50%                   76,771.43                0.00       0.00   
75%                  874,466.92          213,490.62       0.00   
max            4,823,240,240.88      114,805,391.00      31.00   

       DH_DESALOJADOS  DH_DESABRIGADOS  
count        3,756.00         3,756.00  
mean           284.23            39.16  
std          3,956.53           511.98  
min              0.00             0.00  
25%              0.00             0.00  
50%              0.00             0.00  
75%             16.00             0.00  
max        165,000.00        22,00

## 6. Validação da Chave de Cruzamento (Código IBGE)
Para garantir que o nosso futuro cruzamento com a base do MapBiomas (Nível 2) não vai quebrar, precisamos garantir que todos os municípios nesta base possuem o Código IBGE no padrão de 7 dígitos.

In [ ]:
# Converte a coluna de código para texto e mede o tamanho (quantidade de caracteres) de cada linha
tamanho_ibge = df_recorte['Cod_IBGE_Mun'].astype(str).str.len()

print("--- TAMANHO DOS CÓDIGOS IBGE ---")
print(tamanho_ibge.value_counts())
# Se o resultado for apenas "7", significa que 100% da base está no formato correto para o Join!

--- TAMANHO DOS CÓDIGOS IBGE ---
Cod_IBGE_Mun
7    3756
Name: count, dtype: int64


## 7. Diagnóstico de Negócio: Os Falsos "Zero Reais"
Nossa pergunta principal do projeto (MUST) soma os danos materiais e públicos. Porém, como o S2iD depende da autodeclaração das prefeituras, muitos municípios registram o desastre mas não preenchem a estimativa em dinheiro. Vamos descobrir quantos desastres estão "zerados" na nossa base.

In [ ]:
# Filtra as linhas onde TANTO o dano material QUANTO o prejuízo público são exatamente zero
desastres_zerados = df_impacto_numerico[
    (df_impacto_numerico['DM_total_danos_materiais'] == 0) &
    (df_impacto_numerico['PEPL_total_publico'] == 0)
]

percentual_zerados = (len(desastres_zerados) / len(df_recorte)) * 100

print("--- REGISTROS SEM VALOR FINANCEIRO DECLARADO ---")
print(f"Total de desastres com R$ 0,00 declarados: {len(desastres_zerados)} ocorrências.")
print(f"Isso representa {percentual_zerados:.1f}% do nosso recorte.")
print("Nota para o Data Card: Municípios com valores zerados podem cair no final do nosso ranking, não por falta de danos, mas por falta de preenchimento institucional.")

--- REGISTROS SEM VALOR FINANCEIRO DECLARADO ---
Total de desastres com R$ 0,00 declarados: 1407 ocorrências.
Isso representa 37.5% do nosso recorte.
Nota para o Data Card: Municípios com valores zerados podem cair no final do nosso ranking, não por falta de danos, mas por falta de preenchimento institucional.


## 8. Teste de Sanidade (Top Outlier)
Encontramos um valor máximo superior a 4 Bilhões de Reais em um único registro. Vamos identificar que município e evento foi esse para garantir que o número faz sentido no mundo real e não é um erro de digitação do sistema.

In [ ]:
# O comando nlargest(1) pega a linha com o maior valor na coluna especificada
maior_desastre = df_recorte.nlargest(1, 'DM_total_danos_materiais')

print("--- O MAIOR DESASTRE DA BASE (EM DANOS MATERIAIS) ---")
print(maior_desastre[['Nome_Municipio', 'Sigla_UF', 'Data_Evento', 'descricao_tipologia', 'DM_total_danos_materiais']])

--- O MAIOR DESASTRE DA BASE (EM DANOS MATERIAIS) ---
      Nome_Municipio Sigla_UF Data_Evento descricao_tipologia  \
68750   São Leopoldo       RS  27/04/2024     Chuvas Intensas   

       DM_total_danos_materiais  
68750          4,823,240,240.88  


## 9. Exportação do Dataset Filtrado
Criação da métrica principal de custo (Danos Materiais + Prejuízos Públicos) e exportação do arquivo CSV contendo apenas o recorte de desastres hidrológicos da Região Sul (2021-2025). Este é o arquivo que será utilizado no painel.

In [ ]:
# 9. Exportação do Dataset Filtrado
# Garantir que as colunas financeiras sejam lidas como números e preencher vazios com 0
df_recorte['DM_total_danos_materiais'] = pd.to_numeric(df_recorte['DM_total_danos_materiais'], errors='coerce').fillna(0)
df_recorte['PEPL_total_publico'] = pd.to_numeric(df_recorte['PEPL_total_publico'], errors='coerce').fillna(0)

# Criar a métrica do MUST
df_recorte['Custo_Dano_Evitado'] = df_recorte['DM_total_danos_materiais'] + df_recorte['PEPL_total_publico']

# Corrigindo sujeira de espaços em branco nos nomes dos municípios ANTES de exportar
df_recorte['Nome_Municipio_Limpo'] = df_recorte['Nome_Municipio'].astype(str).str.strip().str.title()

# Filtrar apenas as colunas que a squad realmente vai usar no painel (COM A VÍRGULA CORRIGIDA)
colunas_exportacao = [
    'Protocolo_S2iD', 'Cod_IBGE_Mun', 'Nome_Municipio_Limpo', 'Sigla_UF',
    'Data_Evento', 'Cod_Cobrade', 'descricao_tipologia', 'Status',
    'DM_total_danos_materiais', 'PEPL_total_publico', 'Custo_Dano_Evitado',
    'DH_MORTOS', 'DH_DESALOJADOS', 'DH_DESABRIGADOS'
]

df_filtrado = df_recorte[colunas_exportacao].copy()

# Gerar o arquivo CSV
nome_arquivo = 'dataset_sul_hidrologico_2021_2025_filtrado.csv'
df_filtrado.to_csv(nome_arquivo, index=False, encoding='utf-8')

print(f"Arquivo '{nome_arquivo}' gerado com sucesso!")
print(f"Tamanho: {df_filtrado.shape[0]} linhas e {df_filtrado.shape[1]} colunas.")

Arquivo 'dataset_sul_hidrologico_2021_2025_filtrado.csv' gerado com sucesso!
Tamanho: 3756 linhas e 14 colunas.


## 10. Diagnóstico de Negócio: Status vs. Valor Financeiro Zerado
Além de saber que 37,5% dos registros estão com valor zerado, precisamos entender **quem** são esses registros. Será que eles se concentram nos desastres que ainda não foram formalmente reconhecidos pelo governo (Status = "Registro"), ou o problema é espalhado igualmente por toda a base? Essa checagem mostra se o "buraco" financeiro está ligado ao andamento do processo burocrático.

In [ ]:
# Reaproveitamos a mesma lógica de "zerado" usada antes:
# um registro é considerado zerado quando NENHUM dos dois valores (danos materiais e prejuízo público) foi preenchido
recorte_numerico = df_recorte[['DM_total_danos_materiais', 'PEPL_total_publico']].apply(pd.to_numeric, errors='coerce')
df_recorte['zerado'] = (recorte_numerico['DM_total_danos_materiais'] == 0) & (recorte_numerico['PEPL_total_publico'] == 0)

# O comando 'crosstab' cruza duas colunas categóricas e conta quantas vezes cada combinação aparece.
# 'normalize="index"' transforma a contagem em porcentagem DENTRO de cada Status
# (ou seja, responde: "de tudo que é Registro, que fatia está zerada?", separado de "de tudo que é Reconhecido, que fatia está zerada?")
tabela_cruzada = pd.crosstab(df_recorte['Status'], df_recorte['zerado'], normalize='index') * 100

print("--- PERCENTUAL DE REGISTROS ZERADOS, POR STATUS ---")
print(tabela_cruzada.round(1))

# Também é útil ver o número absoluto por trás dessa porcentagem, não só o percentual
print("\n--- QUANTIDADE DE REGISTROS POR STATUS (para referência) ---")
print(df_recorte['Status'].value_counts())

--- PERCENTUAL DE REGISTROS ZERADOS, POR STATUS ---
zerado       False  True 
Status                   
Reconhecido  80.30  19.70
Registro     46.70  53.30

--- QUANTIDADE DE REGISTROS POR STATUS (para referência) ---
Status
Registro       1984
Reconhecido    1772
Name: count, dtype: int64
